# 📘 Pydantic for AI Engineers

## 1. What is Pydantic?

Pydantic is a parsing and validation framework for defining specialized Python classes. It leverages standard Python type hints to ensure data conforms to specific structures and types.

* **Model:** A specialized Pydantic class that inherits from `BaseModel`.
* **Fields:** The class attributes defined within a model.
* **The Golden Rule:** Pydantic guarantees that the types and constraints of the resulting model instance match your definitions. It does the heavy lifting of data validation so you do not have to write boilerplate `if not isinstance(val, str):` checks.

## 2. Pydantic vs. Dataclasses

A common point of confusion is whether to use Dataclasses or Pydantic. They look similar syntactically but serve entirely different core purposes.

| Feature | Standard `dataclass` | Pydantic `BaseModel` |
| --- | --- | --- |
| **Core Mechanism** | Code generator (writes `__init__`, `__repr__` for you). | Inheritance framework (inherits from `BaseModel`). |
| **Validation** | ❌ None out-of-the-box. | ✅ Robust, automatic type checking and validation. |
| **Serialization** | ❌ Manual (requires custom dict/JSON methods). | ✅ Built-in (`model_dump()`, `model_dump_json()`). |
| **Deserialization** | ❌ Manual. | ✅ Built-in (`model_validate()`, `model_validate_json()`). |
| **Performance** | Faster (standard Python objects). | Slightly slower, though V2 is powered by Rust core and highly optimized. |

**When to use which?** * Use **Dataclasses** for internal data structures where performance is critical, and data is already trusted/clean.

* Use **Pydantic** whenever data is crossing a boundary (e.g., API requests in FastAPI, parsing LLM outputs in LangChain, reading JSON from a database).

## 3. The Core Lifecycle

Pydantic primarily exists to handle the flow of data in and out of your applications.

1. **Deserialization & Validation (Input):** Taking raw data (like a Dictionary or JSON string) and converting it into a Python object. During this step, Pydantic actively validates types and constraints.
2. **Manipulation:** Working with the data using standard Python object dot-notation (e.g., `user.first_name`).
3. **Serialization (Output):** Converting the Python object back into a Dictionary or JSON string to send over a network (e.g., returning an API response).

---

## 4. Basic Implementation

### Defining and Instantiating a Model

```python
from pydantic import BaseModel, ValidationError

# 1. Defining the Model
class Person(BaseModel):
    first_name: str
    last_name: str
    age: int

# 2. Creating an instance using keyword arguments
p = Person(first_name="Evariste", last_name="Galois", age=20)

# Pydantic automatically generates a readable __str__ and __repr__
print(p)
# Output: first_name='Evariste' last_name='Galois' age=20

# 3. Accessing and Modifying Fields
print(p.first_name) # Evariste
p.age = 21          # Models are mutable by default
```

In [1]:
from pydantic import BaseModel

In [3]:
# Defining the Model
class Person(BaseModel):
    first_name : str
    last_name : str
    age : int

In [4]:
p = Person(first_name="Vijaya Simha", last_name="M", age=20)

In [9]:
try :
    q= Person(first_name = "user", last_name = "s", age = "twenty")
except Exception as e:
    print(e)

1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


In [11]:
try :
    q= Person(first_name = "user", last_name = "s", age = "20")
except Exception as e:
    print(e)

In [12]:
print(q)

first_name='user' last_name='s' age=20


### Handling Validation Errors

Pydantic collects **all** validation errors before raising an exception, rather than failing at the first bad field. This is crucial for returning complete error payloads in APIs.

```python
try:
    # Missing first_name and age, plus passing a list instead of a string
    bad_person = Person(last_name=["Galois"])
except ValidationError as e:
    print(e)
    """
    Output will list ALL errors:
    - first_name: Field required
    - last_name: Input should be a valid string
    - age: Field required
    """

```

### Deserializing from Dictionaries

In the real world, you rarely instantiate models using keyword arguments. You usually parse raw dictionary data (e.g., from an API payload).

```python
raw_data = {
    "first_name": "Alan",
    "last_name": "Turing",
    "age": 41
}

# Load and validate dictionary data
turing = Person.model_validate(raw_data)

```

---

## 5. Advanced Concepts

As a senior AI engineer, you will need to know these production-level Pydantic features, especially when working with LangChain or Fast API.

### A. Validating Assignments (Fixing the Mutability Loophole)

By default, Pydantic only validates data *upon creation*. If you bypass validation by assigning an invalid type later (e.g., `p.age = "twenty"`), Pydantic ignores it. You can enforce validation on reassignment using `ConfigDict`.

```python
from pydantic import BaseModel, ConfigDict

class StrictPerson(BaseModel):
    model_config = ConfigDict(validate_assignment=True) # Forces validation on changes
    
    first_name: str
    age: int

p = StrictPerson(first_name="Ada", age=36)

try:
    p.age = "thirty-six" # This will now raise a ValidationError!
except ValidationError as e:
    print("Caught bad assignment!")

```

### B. The `Field` Function for Advanced Constraints

Type hints alone aren't always enough. What if `age` must be greater than 0? We use the `Field` function.

```python
from pydantic import BaseModel, Field

class Employee(BaseModel):
    name: str
    # Enforce constraints: greater than 18, less than 100
    age: int = Field(gt=18, lt=100, description="The employee's age") 
    # Use aliases if your incoming JSON has ugly keys (like specific database columns)
    department_id: str = Field(alias="dept_id_xyz")

raw_json = {"name": "Grace", "age": 45, "dept_id_xyz": "ENG-01"}
emp = Employee.model_validate(raw_json)

```

### C. Nested Models (Crucial for AI/LLM Outputs)

When asking an LLM for structured output (e.g., via LangChain's `with_structured_output`), the data is often deeply nested. Pydantic handles nested models seamlessly.

```python
from typing import List

class ToolCall(BaseModel):
    function_name: str
    arguments: dict

class LLMResponse(BaseModel):
    summary: str
    confidence_score: float = Field(ge=0.0, le=1.0)
    tools: List[ToolCall] # Nesting another Pydantic model

# Pydantic will recursively validate the tools list and its contents!

```

### D. Serialization (Exporting Data)

Once your application is done processing the object, you need to turn it back into raw data.

```python
# Convert to a dictionary
person_dict = p.model_dump()

# Convert directly to a JSON string (great for API responses)
person_json = p.model_dump_json(indent=2)

```